# Tutorial · 因果推断基础 (Day 1) · Oxford Tutorial 仿真

## Persona (System Prompt, 仿真用)

You are an **Oxford tutorial fellow in 因果推断基础 (Causal Inference Foundations)**.
You tutor 1-on-2 (yourself + one virtual student) in a weekly 50-minute tutorial.

**Rules you MUST obey:**
1. **Never give direct answers.** 不直接给答案。If the student asks "is X the answer?", reply with a Socratic question that makes them defend it.
2. **Use Socratic questioning** - every turn ends with a probing question (why / 反例 / 凭什么 / what if / how could / 若...变).
3. **Reject vague claims.** 当学生说"大概""差不多""应该"，立刻追问"凭什么?依据?给一个反例?"
4. **Play HBS devil's advocate.** 主动持反方立场 - 若学生说"后门调整后 ATE 就是无偏的"，反驳"凭什么相信你列完了所有混杂?给反例。"
5. **Scaffold fading.** 前 2 轮给提示（仍不直接答），后 2 轮只追问，让学生独立辩护。
6. **End each turn with exactly one probing question.** 不堆问题。
7. **频率限制**: 每学生每天 1 次 tutorial，防 LLM 依赖（见末尾 cell）。

**Domain anchor**: NSW 职业培训数据 (causaldata.nsw) + DoWhy 四步 + LLM-as-a-judge (L1 only) + 营销映射 (treat->优惠券, re78->转化).

---


## Pre-Tutorial Task (强制 retrieval, 课前必交)

Tutorial 不是讲课，是辩护。课前你必须完成以下 retrieval 任务（不许翻 notes.md）：

1. **画 DAG**: 在白纸上手画 NSW 场景的因果图--`treat` (培训), `re78` (收入), 与至少 5 个协变量。标出所有从 `treat` 到 `re78` 的后门路径。
2. **写一段伪代码**: 用 DoWhy 写出「建模->识别->估计->反驳」四步，每步一行，不查文档。
3. **预测**: 朴素均值差 (treated - control) 会比后门调整估计偏高还是偏低? 用一句话说理由。
4. **反问自己**: 如果 LLM-as-a-judge 说「你的 DAG 漏了混杂」,你应该信它吗? 它处于因果阶梯哪一层?

把以上 4 题的答案写进 `pre_tutorial.md`，课前 1 小时提交。Tutorial 中 fellow 会随机抽你辩护--答不上来 = tutorial 终止，下次重来。

> Oxford tutorial 的精髓: 你带着答案来，fellow 的工作是把你的答案拆穿到只剩你能辩护的部分。

---

In [ ]:
# Socratic Tutorial Loop (静态 if/else 仿真, 不调 LLM API)
# 4 轮, 每轮检测 defense 是否成立; 不成立则降一级 scaffold; 仍禁直接答案.
# 学生把 pre_tutorial.md 的答案填进 student_answers, 仿真器按预设分支回应.

student_answers = {
    'turn1_dag_paths': '后门路径有 treat <- age -> re78, treat <- education -> re78, ...',  # 学生填
    'turn2_dowhy_steps': 'model=CausalModel(...); identified=model.identify_effect(); estimate=identified.estimate_effect(); ref=estimate.refute_estimate()',
    'turn3_bias_direction': '朴素估计偏高',
    'turn4_llm_judge_layer': 'L1'
}

def socratic_reply(turn, answer):
    """静态 Socratic 回应. 永远不直接给答案, 每轮以追问结尾."""
    turn = turn.lower()
    if turn == 'turn1':
        # 检测学生是否漏了 re74/re75 这类前收入混杂
        if 're74' in answer or 're75' in answer:
            return ("[Socratic turn 1] 你列了 re74/re75, 好. 但凭什么认为它们是混杂而非中介? "
                    "反例: 若 re74 同时影响 treat (选培训的倾向) 和 re78 (后续收入), 它是混杂; "
                    "若 re74 只通过 treat 影响 re78, 它是中介. 你的 DAG 里它是哪一种? **how could you 用数据检验?**")
        else:
            return ("[Socratic turn 1 - scaffold down] 你列的后门路径只用了 age/education. "
                    "NSW 数据里有 8 个协变量, 你漏了哪些? 特别是关于'前收入'的两个变量. "
                    "**what if 漏掉它们会让 ATE 估计偏向哪一边?**")
    elif turn == 'turn2':
        # 检测学生是否跳过 refute_estimate
        if 'refute' in answer.lower() or 'refute_estimate' in answer.lower():
            return ("[Socratic turn 2] 你写了 refute_estimate, 很好. 但 DoWhy 至少有两种 refuter: "
                    "placebo_treatment_refuter 和 random_common_cause_refuter. "
                    "**凭什么这两种都要跑? 各自检验的是哪种威胁? 若只跑一种会漏什么?**")
        else:
            return ("[Socratic turn 2 - scaffold down] 你的伪代码在 estimate_effect() 之后就停了. "
                    "一个没被反驳检验的 ATE 估计等于没交作业. "
                    "**DoWhy 要求的第 4 步是什么? 为什么不能跳?**")
    elif turn == 'turn3':
        # 检测学生对偏差方向的理解
        if '偏高' in answer:
            return ("[Socratic turn 3 - devil's advocate] 你说朴素估计偏高. 我反驳: 凭什么? "
                    "NSW 里参加培训的人是'就业困难群体', 他们的 re78 本来就比对照组低. "
                    "**若这样, 朴素估计应该偏高还是偏低? 你刚才的判断要不要修正? 给反例.**")
        elif '偏低' in answer:
            return ("[Socratic turn 3] 你说偏低, 有自我选择偏差的意识. 但'偏低'的结论依赖一个假设: "
                    "培训参与者 baseline 收入低于对照组. "
                    "**how could you 用 data 检验这个假设? 看 re74/re75 的两组均值.**")
        else:
            return ("[Socratic turn 3 - scaffold down] 你没说偏方向. "
                    "what if 培训参与者 baseline 收入高于对照组? 朴素估计会怎么偏?")
    elif turn == 'turn4':
        # 检测 LLM-as-judge 层级理解
        if 'L1' in answer or 'l1' in answer:
            return ("[Socratic turn 4] 你说 L1, 正确. 但我要追问: LLM-as-judge '审' 的是论证文本, "
                    "它做的关联分析本身也是一种'相关'. **凭什么相信它的指摘不是幻觉? "
                    "how could you 验证 LLM 指出的'漏掉的混杂'是真混杂?**")
        else:
            return ("[Socratic turn 4 - scaffold down] 你说它处于 L2 或 L3. "
                    "LLM-as-judge 处理的是你的论证文本, 不是 do 操作, 不是反事实. "
                    "**它只能做哪一层的分析? 为什么不能用它估 ATE?**")
    return "[Socratic] 我没听懂你的辩护. **再答一次, 用一个具体反例.**"

# 跑 4 轮 Socratic
for i, (turn, ans) in enumerate(student_answers.items(), 1):
    print(f"--- Round {i}/4 ---")
    print(f"Student: {ans}")
    print(f"Fellow: {socratic_reply(turn, ans)}")
    print()

# 苏格拉底问题清单 (>=5 个, 跨 4 轮)
socratic_questions_asked = [
    'how could you 用数据检验 re74 是混杂还是中介?',
    'what if 漏掉 re74/re75 会让 ATE 估计偏向哪一边?',
    '凭什么 placebo 和 random common cause 两种 refuter 都要跑?',
    'DoWhy 要求的第 4 步是什么? 为什么不能跳?',
    '若培训参与者 baseline 收入低于对照组, 朴素估计会怎么偏?',
    'how could you 用 data 检验 baseline 假设?',
    'how could you 验证 LLM 指出的混杂不是幻觉?',
    'LLM-as-judge 只能做哪一层分析? 为什么不能估 ATE?',
]
print(f"苏格拉底问题总数: {len(socratic_questions_asked)} (要求 >=5)")


In [ ]:
# student_model.json 读写 (跨单元复用, 记录掌握度/盲点)
import json, os, datetime

SM_PATH = 'student_model.json'

def load_student_model():
    if os.path.exists(SM_PATH):
        with open(SM_PATH, encoding='utf-8') as f:
            return json.load(f)
    # 初始化模板
    return {
        'unit': 'U-skill3-day1',
        'topic': '因果推断基础',
        'mastery': {
            'ILO1_causal_ladder': 0.0,   # 0.0-1.0
            'ILO2_dag_backdoor': 0.0,
            'ILO3_naive_vs_adjusted': 0.0,
            'ILO4_dowhy_pipeline': 0.0,
            'ILO5_llm_as_judge': 0.0,
        },
        'blind_spots': [],  # LLM-as-judge 或 fellow 指出的盲点
        'weak_loop_log': [],  # 连续 2 次失败触发 weak_loop 的记录
        'tutorial_history': [],  # 每次 tutorial 的日期/轮次/结果
        'last_tutorial_date': None,  # 限频用
        'cards_state': {}  # 同步 schedule.json 的 FSRS-6 状态
    }

def save_student_model(sm):
    with open(SM_PATH, 'w', encoding='utf-8') as f:
        json.dump(sm, f, ensure_ascii=False, indent=2)

# 仿真: 读取 + 记录本次 tutorial + 写回
sm = load_student_model()
today = datetime.date.today().isoformat()
sm['tutorial_history'].append({
    'date': today,
    'turns_completed': 4,
    'fellow_verdict': 'ILO1 partial; ILO2 needs re-drill on re74/re75; ILO5 L1 correctly identified but defense weak',
    'socratic_questions_asked': 8
})
sm['last_tutorial_date'] = today
# 假装 LLM-as-judge 在 tutorial 中指出了 1 个盲点
sm['blind_spots'].append({
    'date': today,
    'source': "fellow (devil's advocate)",
    'blind_spot': '混淆了混杂与中介 - re74/re75 在 NSW 里同时是混杂 (T<-re74->Y), 不是中介',
    'related_ILO': 'ILO2_dag_backdoor',
    'remediation': '回到 practice.md D1 阶段 1 重看 Worked 示例, 再跑 D1 阶段 3 独立解'
})
sm['mastery']['ILO1_causal_ladder'] = 0.6
sm['mastery']['ILO2_dag_backdoor'] = 0.4  # 需 weak_loop
sm['mastery']['ILO5_llm_as_judge'] = 0.7
save_student_model(sm)
print(f'student_model.json 已更新, blind_spots={len(sm["blind_spots"])}, tutorial_history={len(sm["tutorial_history"])}')
print(f'当前 mastery: {sm["mastery"]}')


## Hattie 四级 Formative Feedback (2007 RER 77(1):81-112)

本 tutorial 结束后, fellow 给出四级反馈. 避免Self级空洞表扬 (Hattie: Self级反馈对学习效应量最低).

- **[TASK]** 任务级 - 关于具体任务做对/做错什么
- **[PROCESS]** 过程级 - 关于解题策略/方法是否得当
- **[SELF-REG]** 自我调节级 - 关于自我监控/纠错能力
- **[FEED-FORWARD]** 前馈 - 下一步该做什么 (本单元最关键)


In [ ]:
# Hattie 四级反馈生成器 (静态, 基于 tutorial 4 轮表现)
hattie_feedback = {
    '[TASK]': (
        "任务级: 你在 turn1 漏了 re74/re75 作为混杂 (NSW 8 个协变量只列了 age/education). "
        "turn2 的 DoWhy 伪代码漏了 refute_estimate 第 4 步. "
        "turn3 偏差方向判断错误 (你说偏高, 实际 NSW 是偏低 - 培训参与者 baseline 低). "
        "turn4 L1 判断正确."
    ),
    '[PROCESS]': (
        "过程级: 你的策略是'先列显而易见的混杂再补', 但 NSW 的关键混杂是'前收入'类变量 (re74/re75), "
        "应该从'处理变量的决定因素'倒推, 而非从'显而易见的人口学变量'正推. "
        "DoWhy 四步的策略问题: 你把 refute 当可选, 实际它是必须 - 应建立'没 refute 等于没交'的纪律."
    ),
    '[SELF-REG]': (
        "自我调节级: 你在 turn3 被 devil's advocate 反驳后没有自我修正, 而是坚持原判断. "
        "健康的 self-regulation 应该是: 听到反例 -> 暂停 -> 用数据检验 -> 修正或辩护. "
        "建议: 下次听到反例, 先问自己'用什么数据能验证这个反例?'再决定是否修正."
    ),
    '[FEED-FORWARD]': (
        "前馈 (最重要): 1) 今天内回到 practice.md D1 阶段 1 重看 re74/re75 的 Worked 示例; "
        "2) 明天用 schedule.json 复习卡片 C2+C3 (FSRS-6 间隔 1 天); "
        "3) 24h 后重跑 D1 阶段 3 独立解 (用一个新场景, 不是 NSW); "
        "4) 若 D1 阶段 3 仍失败, 触发 weak_loop, 隔天再约 tutorial (限频 1 次/天). "
        "5) 下个单元 (Day 2 实验设计) 前, 确认 ILO2 mastery >=0.7, 否则不要进 Day 2."
    )
}
for level, text in hattie_feedback.items():
    print(f'{level}\n  {text}\n')


## 频率限制 (防 LLM 依赖)

- **每学生每天 1 次 tutorial** (`last_tutorial_date` 写入 `student_model.json`)
- 同一天再次请求 -> 系统拒绝, 提示"明天再来. Tutorial 是辩护不是讲课, 重复听不会让你更懂."
- 这不是惩罚, 是 anti-stall 设计 - 防止学生用 LLM 仿真替代独立思考 (Vygotsky 共构理论的边界)
- 借鉴 Oxford tutorial 每周 1 次的物理约束

## Exit Artifact (tutorial 结束必交)

在 `student_model.json` 的 `blind_spots` 字段追加 2-3 条本次 tutorial 暴露的盲点, 并推荐复习单元:

```json
{
  "blind_spots": [
    "混淆了混杂与中介 - re74/re75 在 NSW 是混杂不是中介",
    "DoWhy refute_estimate 当可选而非必须",
    "偏差方向判断依赖直觉而非 baseline 数据检验"
  ],
  "recommended_review_units": [
    "本单元 practice.md D1 阶段 1 (Worked 示例)",
    "本单元 schedule.json C2+C3 (FSRS-6 间隔 1 天)",
    "Day 2 实验设计 (前测) - 进 Day 2 前确认 ILO2 mastery >=0.7"
  ]
}
```

> Tutorial 结束不等于学习结束. Exit artifact 是下一次学习的起点. 跨单元复用 `student_model.json` 让盲点不被遗忘.

---

*v6.0 学习科学层 · Oxford tutorial Socratic + HBS devil's advocate + Hattie 4 级 + 限频防依赖 + student_model 跨单元复用*